# Temperature

In [159]:
# Land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)  # collapse the single 'land' region

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_base_19712000_annual.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_present_19812010_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_future585_20712100_annual.nc")


# --- Build masks on each dataset's own grid (safe even if grids match) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Apply land-only masking (set ocean grid cells to NaN) ---
base_annual["tas"] = base_annual["tas"].where(land_mask_base)
present_annual["tas"] = present_annual["tas"].where(land_mask_present)

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d_base = weights.broadcast_like(base_reg["tas"].isel(year=0)).where(mask_reg_base)
    weights_2d_present = weights.broadcast_like(present_reg["tas"].isel(year=0)).where(mask_reg_present)

    # Calculate annual means for baseline period (1971-2000), land only
    base_annual_means = (base_reg["tas"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])

    # Calculate annual means for present period (1981-2010), land only
    present_annual_means = (present_reg["tas"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

    # Calculate baseline climatology (1971-2000 mean) - keep in Kelvin for percentage calculation
    baseline_climatology_K = base_annual_means.mean()

    # Calculate temperature changes for each year in 1981-2010 relative to baseline
    temperature_changes = present_annual_means - baseline_climatology_K

    # Convert absolute changes to Celsius for reporting
    temperature_changes_celsius = temperature_changes

    # Mean and standard deviation of temperature changes in Celsius
    mean_change_celsius = temperature_changes_celsius.mean().item()
    std_change_celsius = temperature_changes_celsius.std(ddof=1).item()

    # Calculate percentage changes using Kelvin baseline
    pct_change_mean = (mean_change_celsius / baseline_climatology_K.item()) * 100
    pct_change_std = (std_change_celsius / baseline_climatology_K.item()) * 100

    results.append({
        "Region": region,
        "Mean_Change_C": mean_change_celsius,
        "Std_Change_C": std_change_celsius,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Temperature Changes: 1981-2010 relative to 1971-2000 (Land only)")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(°C)':>12}{'(K basis)':>10}{'(°C)':>12}{'(K basis)':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_C']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_C']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average temperature change over 30 years (1981-2010) in °C")
print(f"Mean %: Percentage of mean temperature change relative to baseline (using Kelvin)")
print(f"Std Change: Standard deviation of the 30 annual temperature changes in °C")
print(f"Std %: Percentage of standard deviation relative to baseline (using Kelvin)")

Temperature Changes: 1981-2010 relative to 1971-2000 (Land only)
Region               Mean Change    Mean %  Std Change     Std %
                            (°C) (K basis)        (°C) (K basis)
----------------------------------------------------------------------
Global                     4.811      1.66       0.675      0.23
Tropics                    4.311      1.45       0.685      0.23
Subtropics_N               4.964      1.71       0.687      0.24
Subtropics_S               3.686      1.26       0.595      0.20
Mid_Latitudes_N            5.792      2.08       0.810      0.29
Mid_Latitudes_S            2.287      0.80       0.462      0.16

Mean Change: Average temperature change over 30 years (1981-2010) in °C
Mean %: Percentage of mean temperature change relative to baseline (using Kelvin)
Std Change: Standard deviation of the 30 annual temperature changes in °C
Std %: Percentage of standard deviation relative to baseline (using Kelvin)


# Precipitable water

In [68]:
# land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/prw/CMCC-ESM2_prw_base_19712000_annual.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/prw/CMCC-ESM2_prw_present_19812010_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/prw/CMCC-ESM2_prw_future126_20712100_annual.nc")


# --- Build masks on each dataset's own grid ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Apply land-only masking (set ocean grid cells to NaN) ---
base_annual["prw"] = base_annual["prw"].where(land_mask_base)
present_annual["prw"] = present_annual["prw"].where(land_mask_present)

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d_base = weights.broadcast_like(base_reg["prw"].isel(year=0)).where(mask_reg_base)
    weights_2d_present = weights.broadcast_like(present_reg["prw"].isel(year=0)).where(mask_reg_present)

    # Calculate annual means for baseline period (1971-2000), land only
    base_annual_means = (base_reg["prw"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])

    # Calculate annual means for present period (1981-2010), land only
    present_annual_means = (present_reg["prw"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()

    # Calculate precipitable water changes for each year in 1981-2010 relative to baseline
    prw_changes = present_annual_means - baseline_climatology

    # Convert from kg/m2 to mm (1 kg/m2 = 1 mm)
    baseline_mm = baseline_climatology.item()
    prw_changes_mm = prw_changes

    # Mean and standard deviation of precipitable water changes in mm
    mean_change_mm = prw_changes_mm.mean().item()
    std_change_mm = prw_changes_mm.std(ddof=1).item()

    # Calculate percentage changes
    pct_change_mean = (mean_change_mm / baseline_mm) * 100
    pct_change_std = (std_change_mm / baseline_mm) * 100

    results.append({
        "Region": region,
        "Mean_Change_mm": mean_change_mm,
        "Std_Change_mm": std_change_mm,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitable Water Changes: 1981-2010 relative to 1971-2000 (Land only)")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm)':>12}{'':>10}{'(mm)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mm']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mm']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitable water change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean precipitable water change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitable water changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitable Water Changes: 1981-2010 relative to 1971-2000 (Land only)
Region               Mean Change    Mean %  Std Change     Std %
                            (mm)                  (mm)          
----------------------------------------------------------------------
Global                     4.026     17.51       0.477      2.07
Tropics                    5.068     15.00       0.823      2.43
Subtropics_N               3.555     20.84       0.598      3.50
Subtropics_S               2.298     10.24       0.701      3.12
Mid_Latitudes_N            3.446     27.59       0.306      2.45
Mid_Latitudes_S            1.573     10.47       0.330      2.20

Mean Change: Average precipitable water change over 30 years (1981-2010)
Mean %: Percentage of mean precipitable water change relative to baseline
Std Change: Standard deviation of the 30 annual precipitable water changes
Std %: Percentage of standard deviation relative to baseline


# Precipitation

In [109]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_base_19712000_annual.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_present_19812010_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_future126_20712100_annual.nc")

# --- Build masks on each dataset's own grid ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Apply land-only masking (set ocean grid cells to NaN) ---
base_annual["pr"] = base_annual["pr"].where(land_mask_base)
present_annual["pr"] = present_annual["pr"].where(land_mask_present)

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d_base = weights.broadcast_like(base_reg["pr"].isel(year=0)).where(mask_reg_base)
    weights_2d_present = weights.broadcast_like(present_reg["pr"].isel(year=0)).where(mask_reg_present)

    # Calculate annual means for baseline period (1971-2000), land only
    base_annual_means = (base_reg["pr"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])

    # Calculate annual means for present period (1981-2010), land only
    present_annual_means = (present_reg["pr"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()

    # Calculate precipitation changes for each year in 1981-2010 relative to baseline
    precipitation_changes = present_annual_means - baseline_climatology

    # Convert from kg/m²/s to mm/day (multiply by 86400)
    baseline_mmday = baseline_climatology.item() * 86400
    precipitation_changes_mmday = precipitation_changes * 86400

    # Mean and standard deviation of precipitation changes in mm/day
    mean_change_mmday = precipitation_changes_mmday.mean().item()
    std_change_mmday = precipitation_changes_mmday.std(ddof=1).item()

    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100

    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitation Changes: 1981-2010 relative to 1971-2000 (Land only)")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitation change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean precipitation change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitation changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitation Changes: 1981-2010 relative to 1971-2000 (Land only)
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
----------------------------------------------------------------------
Global                     0.181      6.81       0.078      2.93
Tropics                    0.187      5.03       0.161      4.31
Subtropics_N               0.174      9.53       0.099      5.42
Subtropics_S              -0.064     -2.98       0.183      8.56
Mid_Latitudes_N            0.242     13.50       0.061      3.40
Mid_Latitudes_S            0.004      0.15       0.135      5.18

Mean Change: Average precipitation change over 30 years (1981-2010)
Mean %: Percentage of mean precipitation change relative to baseline
Std Change: Standard deviation of the 30 annual precipitation changes
Std %: Percentage of standard deviation relative to baseline


# Evapotranspiration

In [154]:
# Land only = evapotranspiration

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_base_19712000_annual.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/hfls/CMCC-CM2-SR5_hfls_present_19812010_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_future126_20712100_annual.nc")

# --- Build masks on each dataset's own grid ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Apply land-only masking (set ocean grid cells to NaN) ---
base_annual["hfls"] = base_annual["hfls"].where(land_mask_base)
present_annual["hfls"] = present_annual["hfls"].where(land_mask_present)

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d_base = weights.broadcast_like(base_reg["hfls"].isel(year=0)).where(mask_reg_base)
    weights_2d_present = weights.broadcast_like(present_reg["hfls"].isel(year=0)).where(mask_reg_present)

    # Calculate annual means for baseline period (1971-2000), land only
    base_annual_means = (base_reg["hfls"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])

    # Calculate annual means for present period (1981-2010), land only
    present_annual_means = (present_reg["hfls"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()

    # Calculate latent heat flux changes for each year in 1981-2010 relative to baseline
    hfls_changes = present_annual_means - baseline_climatology

    # Convert from W/m² to mm/day equivalent evaporation
    baseline_mmday = baseline_climatology.item() * 86400 / (2.45 * 1e6)  # W/m² to mm/day
    evaporation_changes_mmday = hfls_changes * 86400 / (2.45 * 1e6)  # W/m² to mm/day

    # Mean and standard deviation of evaporation changes in mm/day
    mean_change_mmday = evaporation_changes_mmday.mean().item()
    std_change_mmday = evaporation_changes_mmday.std(ddof=1).item()

    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100

    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Latent Heat Flux (Evaporation) Changes: 1981-2010 relative to 1971-2000 (Land only)")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average evaporation change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean evaporation change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual evaporation changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Latent Heat Flux (Evaporation) Changes: 1981-2010 relative to 1971-2000 (Land only)
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
----------------------------------------------------------------------
Global                     0.054      3.46       0.017      1.08
Tropics                    0.019      0.86       0.027      1.25
Subtropics_N               0.092      8.66       0.037      3.45
Subtropics_S              -0.043     -2.83       0.073      4.79
Mid_Latitudes_N            0.106     10.23       0.016      1.51
Mid_Latitudes_S            0.029      1.98       0.051      3.52

Mean Change: Average evaporation change over 30 years (1981-2010)
Mean %: Percentage of mean evaporation change relative to baseline
Std Change: Standard deviation of the 30 annual evaporation changes
Std %: Percentage of standard deviation relative to baseline


# Soil Moisture

In [38]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls/mrsos scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_base_19712000_annual.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_present_19812010_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrsos/MIROC6_mrsos_future126_20712100_annual.nc")

# --- Build masks on each dataset's own grid ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
#present_annual = extract_annual(ds_present).sel(year=slice(1981, 2010))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Apply land-only masking (set ocean grid cells to NaN) ---
base_annual["mrsos"] = base_annual["mrsos"].where(land_mask_base)
present_annual["mrsos"] = present_annual["mrsos"].where(land_mask_present)

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d_base = weights.broadcast_like(base_reg["mrsos"].isel(year=0)).where(mask_reg_base)
    weights_2d_present = weights.broadcast_like(present_reg["mrsos"].isel(year=0)).where(mask_reg_present)

    # Calculate annual means for baseline period (1971-2000), land only
    base_annual_means = (base_reg["mrsos"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])

    # Calculate annual means for present period (1981-2010), land only
    present_annual_means = (present_reg["mrsos"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()

    # Calculate soil moisture changes for each year in 1981-2010 relative to baseline
    mrsos_changes = present_annual_means - baseline_climatology

    # Convert from kg/m² to m³/m³ (volumetric water content)
    soil_depth = 0.1  # meters (10 cm soil layer)
    water_density = 1000  # kg/m³

    baseline_m3m3 = baseline_climatology.item() / (water_density * soil_depth)
    soil_moisture_changes_m3m3 = mrsos_changes / (water_density * soil_depth)

    # Mean and standard deviation of soil moisture changes in m³/m³
    mean_change_m3m3 = soil_moisture_changes_m3m3.mean().item()
    std_change_m3m3 = soil_moisture_changes_m3m3.std(ddof=1).item()

    # Calculate percentage changes
    pct_change_mean = (mean_change_m3m3 / baseline_m3m3) * 100
    pct_change_std = (std_change_m3m3 / baseline_m3m3) * 100

    results.append({
        "Region": region,
        "Mean_Change_m3m3": mean_change_m3m3,
        "Std_Change_m3m3": std_change_m3m3,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std,
        "Baseline_m3m3": baseline_m3m3
    })

# --- Print results ---
print("Soil Moisture Changes: 1981-2010 relative to 1971-2000 (Land only)")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(m³/m³)':>12}{'':>10}{'(m³/m³)':>12}{'':>10}")
print("-" * 80)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_m3m3']:>12.6f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_m3m3']:>12.6f}{res['Pct_Change_Std']:>10.2f}")

print(f"Mean %: Percentage of mean soil moisture change relative to baseline, land only")
print(f"Std Change: Standard deviation of the 30 annual soil moisture changes, land only")
print(f"Std %: Percentage of standard deviation relative to baseline, land only")
print(f"Note: Assumes 10 cm soil depth and water density of 1000 kg/m³")
print(f"Conversion formula: (kg/m²) / (1000 kg/m³ × 0.1 m) = m³/m³")

Soil Moisture Changes: 1981-2010 relative to 1971-2000 (Land only)
Region               Mean Change    Mean %  Std Change     Std %
                         (m³/m³)               (m³/m³)          
--------------------------------------------------------------------------------
Global                 -0.004482     -1.96    0.001890      0.83
Tropics                -0.002331     -1.09    0.005023      2.36
Subtropics_N           -0.002996     -1.70    0.003602      2.05
Subtropics_S           -0.002435     -1.42    0.009026      5.28
Mid_Latitudes_N        -0.008526     -3.01    0.001946      0.69
Mid_Latitudes_S        -0.003200     -1.25    0.005470      2.13
Mean %: Percentage of mean soil moisture change relative to baseline, land only
Std Change: Standard deviation of the 30 annual soil moisture changes, land only
Std %: Percentage of standard deviation relative to baseline, land only
Note: Assumes 10 cm soil depth and water density of 1000 kg/m³
Conversion formula: (kg/m²) / (1000 

# Runoff

In [74]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls/mrsos scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrro/CMCC-ESM2_mrro_base_19712000_annual.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_future585_20212050_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrro/CMCC-ESM2_mrro_present_19812010_annual.nc")

# --- Build masks on each dataset's own grid ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(1981, 2010))

# --- Apply land-only masking (set ocean grid cells to NaN) ---
base_annual["mrro"] = base_annual["mrro"].where(land_mask_base)
present_annual["mrro"] = present_annual["mrro"].where(land_mask_present)

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d_base = weights.broadcast_like(base_reg["mrro"].isel(year=0)).where(mask_reg_base)
    weights_2d_present = weights.broadcast_like(present_reg["mrro"].isel(year=0)).where(mask_reg_present)

    # Calculate annual means for baseline period (1971-2000), land only
    base_annual_means = (base_reg["mrro"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])

    # Calculate annual means for present period (2071-2100), land only
    present_annual_means = (present_reg["mrro"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()

    # Calculate runoff changes for each year in 2071-2100 relative to baseline
    mrro_changes = present_annual_means - baseline_climatology

    # Convert from kg/m²/s to mm/day
    baseline_mmday = baseline_climatology.item() * 86400  # 1 kg/m²/s = 86400 mm/day
    runoff_changes_mmday = mrro_changes * 86400  # 1 kg/m²/s = 86400 mm/day

    # Mean and standard deviation of runoff changes in mm/day
    mean_change_mmday = runoff_changes_mmday.mean().item()
    std_change_mmday = runoff_changes_mmday.std(ddof=1).item()

    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100

    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std,
        "Baseline_mmday": baseline_mmday
    })

# --- Print results ---
print("Runoff Changes: 2071-2100 relative to 1971-2000 (Land only)")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 80)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.4f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.4f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average runoff change over 30 years (2071-2100), land only")
print(f"Mean %: Percentage of mean runoff change relative to baseline, land only")
print(f"Std Change: Standard deviation of the 30 annual runoff changes, land only")
print(f"Std %: Percentage of standard deviation relative to baseline, land only")
print(f"Note: mrro represents total runoff (surface + subsurface)")
print(f"Conversion: 1 kg/m²/s = 86400 mm/day")

Runoff Changes: 2071-2100 relative to 1971-2000 (Land only)
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
--------------------------------------------------------------------------------
Global                   -0.0005     -0.03      0.0770      5.00
Tropics                  -0.0074     -0.31      0.1781      7.58
Subtropics_N             -0.0039     -0.31      0.1317     10.60
Subtropics_S             -0.0018     -0.31      0.0740     13.22
Mid_Latitudes_N           0.0105      1.25      0.0447      5.29
Mid_Latitudes_S          -0.0052     -0.30      0.1083      6.22

Mean Change: Average runoff change over 30 years (2071-2100), land only
Mean %: Percentage of mean runoff change relative to baseline, land only
Std Change: Standard deviation of the 30 annual runoff changes, land only
Std %: Percentage of standard deviation relative to baseline, land only
Note: mrro represents total runoff (surface + s